In [ ]:
from pathlib import Path
import os, sys
WORKSPACE = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / 'src/branch_sql_MVP/settings.json').is_file())
sys.path.insert(0, str(WORKSPACE)) if str(WORKSPACE) not in sys.path else None
MVP_ROOT = WORKSPACE / 'src/branch_sql_MVP'
os.chdir(MVP_ROOT)


# Luna component tuning và full-test evaluation

Notebook này là artifact đánh giá độc lập cho `openai/gpt-5.6-luna`. Mọi lựa chọn tham số dùng **dev בלבד**; cấu hình được khóa một lần trước khi chạy toàn bộ 8 kịch bản trên 62 mẫu test. Tiêu chí chọn theo thứ tự: execution accuracy cao nhất, run error thấp nhất, invalid SQL thấp nhất, rồi tổng input/output token thấp nhất.

## Chunk contract được áp dụng

- Một mục business bắt đầu bằng heading `###`, gồm bảng cột và các công thức `Q0`, `Q1`, ... tương ứng, là một parent chunk nguyên vẹn.
- Mỗi câu gold SQL bắt đầu bằng `-- Query:` là một parent chunk riêng. Query người dùng được so khớp với child chunk; khi retrieve sẽ trả lại toàn bộ parent chứa Query, Evidence và SQL.
- Gold SQL chỉ được evaluator đọc sau khi prediction đã đóng băng; không đi vào prompt sinh SQL.

> `R-VES` giữ giá trị null vì repository chưa tích hợp official BIRD R-VES evaluator; notebook không suy diễn R-VES từ execution accuracy.

In [1]:
from __future__ import annotations

import json
from collections import Counter
from pathlib import Path

import pandas as pd
from IPython.display import Markdown, display

from src.branch_sql_MVP.data.catalog import load_benchmark_cases
from src.branch_sql_MVP.eval.text2sql import evaluate_prediction
from src.branch_sql_MVP.settings import load_settings

ROOT = Path.cwd().resolve()
BUNDLE_PATH = ROOT / '.runtime/text2sql_benchmark_multimodel/luna_component_tuning.json'
OLD_BUNDLE_PATH = ROOT / '.runtime/text2sql_benchmark_multimodel/benchmark_bundle.gpt-5.6-luna.json'
bundle = json.loads(BUNDLE_PATH.read_text(encoding='utf-8'))
pd.set_option('display.max_columns', 100)
pd.set_option('display.max_rows', 200)
display(Markdown(f'**Bundle:** `{BUNDLE_PATH}`'))

**Bundle:** `C:\Users\Khanh\Documents\vi-coze\src\branch_sql_MVP\.runtime\text2sql_benchmark_multimodel\luna_component_tuning.json`

## Protocol, model và kiểm tra tính toàn vẹn

In [2]:
protocol = bundle['protocol']
tests = bundle['test_scenarios']
settings = load_settings()
dev_all = load_benchmark_cases(settings.path(settings.paths.dev), 'dev')
dev_lookup = {case.stable_id: case for case in dev_all}
dev_sample = [dev_lookup[item] for item in bundle['dev_stable_ids']]
dev_distribution = Counter((case.db_id, case.difficulty) for case in dev_sample)
difficulty_distribution = Counter(case.difficulty for case in dev_sample)

assert protocol['provider'] == 'openai'
assert protocol['model'] == 'gpt-5.6-luna'
assert len(dev_sample) == 33 and len(set(bundle['dev_stable_ids'])) == 33
assert difficulty_distribution == {'simple': 11, 'moderate': 11, 'challenging': 11}
assert len(dev_distribution) == 33 and set(dev_distribution.values()) == {1}
assert len(tests) == 8
assert all(row['metrics']['cases'] == 62 for row in tests)
assert all(row['metrics']['completed_cases'] == 62 for row in tests)
assert all(row['metrics']['run_error_rate'] == 0 for row in tests)

protocol_view = {**protocol, 'dev_difficulty_distribution': dict(difficulty_distribution)}
display(pd.DataFrame([protocol_view]).T.rename(columns={0: 'value'}))
print('Integrity checks: PASS')

,value
order,"coordinate search on dev only, freeze once, ev..."
selection_metric_order,"[execution_accuracy:max, run_error_rate:min, i..."
dev_total_cases,1534
dev_tuning_cases,33
dev_sampling,"1 case for each database x each difficulty, se..."
test_cases,62
test_scope,all cases
provider,openai
model,gpt-5.6-luna
reasoning_effort,selected on dev


Integrity checks: PASS


## Toàn bộ không gian tham số và kết quả từng thí nghiệm dev

In [3]:
space_rows = []
space_rows.append({'component': 'reasoning_effort', 'grid': bundle['parameter_space']['reasoning_grid']})
for stage in bundle['parameter_space']['context_stages']:
    space_rows.append({'component': stage['name'], 'grid': stage['candidates']})
space_rows.extend([
    {'component': 'repair_bound', 'grid': bundle['parameter_space']['repair_grid']},
    {'component': 'candidate_bound', 'grid': bundle['parameter_space']['candidate_grid']},
])
display(pd.DataFrame(space_rows))

selected_key = {
    'repair_bound': 'repair_bound',
    'candidate_bound': 'candidate_bound',
}
dev_records = []
for stage, rows in bundle['dev_tuning'].items():
    selected = bundle['selected'][selected_key.get(stage, stage)]
    for row in rows:
        metrics = row['metrics']
        proxy = row.get('proxy_metrics', {})
        dev_records.append({
            'component': stage,
            'candidate': row.get('name', row.get('value')),
            'selected': row.get('run_id') == selected.get('run_id'),
            'scenario': row['scenario'],
            'execution_accuracy': metrics['execution_accuracy'],
            'invalid_sql_rate': metrics['invalid_sql_rate'],
            'run_error_rate': metrics['run_error_rate'],
            'mean_repairs': metrics['mean_repairs'],
            'mean_candidates': metrics['mean_candidates'],
            'input_tokens_all_calls': metrics['input_tokens_all_calls'],
            'output_tokens_all_calls': metrics['output_tokens_all_calls'],
            'schema_table_recall': proxy.get('schema_table_recall'),
            'schema_complete_rate': proxy.get('schema_complete_rate'),
            'event_gold_table_recall': proxy.get('event_gold_table_recall'),
            'run_id': row.get('run_id'),
        })
dev_df = pd.DataFrame(dev_records)
display(dev_df.style.format({
    'execution_accuracy': '{:.2%}', 'invalid_sql_rate': '{:.2%}',
    'run_error_rate': '{:.2%}', 'schema_table_recall': '{:.2%}',
    'schema_complete_rate': '{:.2%}', 'event_gold_table_recall': '{:.2%}',
}, na_rep='—'))

,component,grid
0,reasoning_effort,"[low, medium, high]"
1,retrieval_fusion,"[{'name': 'dense_only', 'mode': 'semantic', 's..."
2,reranking,"[{'name': 'rerank_off', 'rerank_enabled': Fals..."
3,retrieval_depth,"[{'name': 'depth_3', 'candidate_k': 8, 'docs_t..."
4,schema_linking,"[{'name': 'schema_compact', 'table_k': 3, 'col..."
5,value_linking,"[{'name': 'values_off', 'value_k': 0, 'value_f..."
6,context_budget,"[{'name': 'context_2k', 'token_budget': 2000},..."
7,event_expansion,"[{'name': 'events_off', 'use_events': False, '..."
8,contextual_selector,"[{'name': 'selector_off', 'contextual_selector..."
9,repair_bound,"[0, 1, 2, 3]"


,component,candidate,selected,scenario,execution_accuracy,invalid_sql_rate,run_error_rate,mean_repairs,mean_candidates,input_tokens_all_calls,output_tokens_all_calls,schema_table_recall,schema_complete_rate,event_gold_table_recall,run_id
0,reasoning_effort,low,False,P1,42.42%,0.00%,0.00%,0.000000,1.000000,282273,6534,—,—,—,dev-component-reasoning-low-b0ea8671f9e0
1,reasoning_effort,medium,False,P1,45.45%,0.00%,0.00%,0.000000,1.000000,282273,8368,—,—,—,dev-component-reasoning-medium-91c34218a63e
2,reasoning_effort,high,True,P1,48.48%,0.00%,0.00%,0.000000,1.000000,282273,13068,—,—,—,dev-component-reasoning-high-8f480f82fbb9
3,retrieval_fusion,dense_only,False,B4,45.45%,0.00%,0.00%,0.000000,1.000000,119636,15805,100.00%,100.00%,—,dev-component-retrieval_fusion-dense_only-a0b285d2f001
4,retrieval_fusion,keyword_only,False,B4,48.48%,0.00%,0.00%,0.000000,1.000000,137959,15123,100.00%,100.00%,—,dev-component-retrieval_fusion-keyword_only-5bfaabebb722
5,retrieval_fusion,hybrid_keyword_0_7,False,B4,51.52%,0.00%,0.00%,0.000000,1.000000,131238,15716,100.00%,100.00%,—,dev-component-retrieval_fusion-hybrid_keyword_0_7-5b846efdcd38
6,retrieval_fusion,hybrid_balanced,True,B4,51.52%,0.00%,0.00%,0.000000,1.000000,127710,15478,100.00%,100.00%,—,dev-component-retrieval_fusion-hybrid_balanced-df14e8ea932d
7,retrieval_fusion,hybrid_dense_0_7,False,B4,48.48%,3.03%,0.00%,0.000000,1.000000,124032,17375,100.00%,100.00%,—,dev-component-retrieval_fusion-hybrid_dense_0_7-38d1cfd5c585
8,reranking,incumbent_before_reranking,True,B4,51.52%,0.00%,0.00%,0.000000,1.000000,127710,15478,100.00%,100.00%,—,dev-component-retrieval_fusion-hybrid_balanced-df14e8ea932d
9,reranking,rerank_off,False,B4,48.48%,0.00%,0.00%,0.000000,1.000000,127710,15334,100.00%,100.00%,—,dev-component-reranking-rerank_off-71b5f18a6b5a


## Cấu hình tốt nhất được khóa trên dev

In [4]:
selection_rows = []
for component, row in bundle['selected'].items():
    if component == 'frozen_context':
        continue
    selection_rows.append({
        'component': component,
        'winner': row.get('name', row.get('value')),
        'execution_accuracy': row['metrics']['execution_accuracy'],
        'invalid_sql_rate': row['metrics']['invalid_sql_rate'],
        'input_tokens': row['metrics']['input_tokens_all_calls'],
        'output_tokens': row['metrics']['output_tokens_all_calls'],
        'run_id': row['run_id'],
    })
display(pd.DataFrame(selection_rows).style.format({
    'execution_accuracy': '{:.2%}', 'invalid_sql_rate': '{:.2%}'
}))
display(Markdown('### Frozen context'))
display(pd.DataFrame(bundle['selected']['frozen_context'].items(), columns=['parameter', 'value']))

,component,winner,execution_accuracy,invalid_sql_rate,input_tokens,output_tokens,run_id
0,reasoning_effort,high,48.48%,0.00%,282273,13068,dev-component-reasoning-high-8f480f82fbb9
1,retrieval_fusion,hybrid_balanced,51.52%,0.00%,127710,15478,dev-component-retrieval_fusion-hybrid_balanced-df14e8ea932d
2,reranking,incumbent_before_reranking,51.52%,0.00%,127710,15478,dev-component-retrieval_fusion-hybrid_balanced-df14e8ea932d
3,retrieval_depth,depth_5,54.55%,6.06%,127710,16183,dev-component-retrieval_depth-depth_5-234a71aaf6ae
4,schema_linking,schema_balanced,57.58%,3.03%,127710,15649,dev-component-schema_linking-schema_balanced-2656d327232c
5,value_linking,incumbent_before_value_linking,57.58%,3.03%,127710,15649,dev-component-schema_linking-schema_balanced-2656d327232c
6,context_budget,incumbent_before_context_budget,57.58%,3.03%,127710,15649,dev-component-schema_linking-schema_balanced-2656d327232c
7,event_expansion,events_expand_12x2,54.55%,0.00%,188137,17613,dev-component-event_expansion-events_expand_12x2-9403f1714683
8,contextual_selector,incumbent_before_contextual_selector,54.55%,0.00%,188137,17613,dev-component-event_expansion-events_expand_12x2-9403f1714683
9,repair_bound,1,51.52%,0.00%,188203,14079,dev-component-repair-1-d0e82e7dec5f


### Frozen context

,parameter,value
0,mode,hybrid
1,candidate_k,16
2,docs_top_k,5
3,semantic_weight,0.5
4,keyword_weight,0.5
5,rrf_k,40
6,rerank_top_k,5
7,rerank_enabled,False
8,rerank_max_length,128
9,rerank_batch_size,8


## Kết quả đầy đủ trên test (8 kịch bản × 62 case)

In [5]:
test_records = []
for row in tests:
    metrics = row['metrics']
    test_records.append({
        'scenario_label': row['label'],
        'workflow': row['scenario'],
        **metrics,
        'run_id': row['run_id'],
    })
test_df = pd.DataFrame(test_records)
display(test_df.style.format({
    'execution_accuracy': '{:.2%}', 'exact_sql_match': '{:.2%}',
    'invalid_sql_rate': '{:.2%}', 'empty_result_rate': '{:.2%}',
    'run_error_rate': '{:.2%}', 'mean_generation_latency_seconds': '{:.2f}',
}))

input_tokens = int(test_df['input_tokens_all_calls'].sum())
output_tokens = int(test_df['output_tokens_all_calls'].sum())
estimated_test_cost_usd = input_tokens / 1_000_000 * 0.20 + output_tokens / 1_000_000 * 1.20
print(f'Test tokens: input={input_tokens:,}, output={output_tokens:,}')
print(f'Estimated test cost at listed Luna rates (without cache discount): ${estimated_test_cost_usd:.3f}')

,scenario_label,workflow,cases,completed_cases,execution_accuracy,exact_sql_match,invalid_sql_rate,empty_result_rate,run_error_rate,mean_generation_latency_seconds,input_tokens,output_tokens,mean_repairs,mean_candidates,r_ves,r_ves_reason,input_tokens_all_calls,output_tokens_all_calls,run_id
0,optimized-luna-P1-full-one-shot,P1,62,62,43.55%,0.00%,1.61%,11.29%,0.00%,4.82,216880,32503,0.000000,1.000000,None,Chưa tích hợp official BIRD R-VES evaluator; không suy diễn từ execution accuracy.,216880,32503,test-optimized-luna-P1-full-one-shot-4a9a02240d34
1,optimized-luna-B4-hybrid-fixed,B4,62,62,43.55%,0.00%,1.61%,11.29%,0.00%,5.91,190162,44984,0.000000,1.000000,None,Chưa tích hợp official BIRD R-VES evaluator; không suy diễn từ execution accuracy.,190162,44984,test-optimized-luna-B4-hybrid-fixed-8e9d7e4b63b0
2,optimized-luna-B4-plus-event-seeds,B5,62,62,48.39%,0.00%,0.00%,9.68%,0.00%,5.27,350421,33769,0.000000,1.000000,None,Chưa tích hợp official BIRD R-VES evaluator; không suy diễn từ execution accuracy.,350421,33769,test-optimized-luna-B4-plus-event-seeds-6b7b4a430bf5
3,optimized-luna-B4-plus-event-expansion,B5,62,62,48.39%,0.00%,1.61%,8.06%,0.00%,6.34,367386,39679,0.000000,1.000000,None,Chưa tích hợp official BIRD R-VES evaluator; không suy diễn từ execution accuracy.,367386,39679,test-optimized-luna-B4-plus-event-expansion-7ad23077b58f
4,optimized-luna-B5-relational-selector,B5,62,62,38.71%,0.00%,0.00%,14.52%,0.00%,5.98,367386,37956,0.000000,1.000000,None,Chưa tích hợp official BIRD R-VES evaluator; không suy diễn từ execution accuracy.,367386,37956,test-optimized-luna-B5-relational-selector-b38a2fc0d35f
5,optimized-luna-B6-repair-relational,B6,62,62,45.16%,0.00%,0.00%,11.29%,0.00%,5.81,367578,35553,0.016129,1.016129,None,Chưa tích hợp official BIRD R-VES evaluator; không suy diễn từ execution accuracy.,373346,36724,test-optimized-luna-B6-repair-relational-def426577b6e
6,optimized-luna-B6-repair-hybrid,B6,62,62,45.16%,0.00%,0.00%,12.90%,0.00%,5.27,190162,33876,0.000000,1.000000,None,Chưa tích hợp official BIRD R-VES evaluator; không suy diễn từ execution accuracy.,190162,33876,test-optimized-luna-B6-repair-hybrid-0f6014baa3bf
7,optimized-luna-G1-adaptive,G1,62,62,50.00%,0.00%,0.00%,6.45%,0.00%,6.13,268400,38268,0.000000,1.000000,None,Chưa tích hợp official BIRD R-VES evaluator; không suy diễn từ execution accuracy.,279055,48211,test-optimized-luna-G1-adaptive-8b4a8db93a9b


Test tokens: input=2,334,798, output=307,702
Estimated test cost at listed Luna rates (without cache discount): $0.836


## So sánh với Luna baseline cũ (nếu artifact tồn tại)

In [6]:
if OLD_BUNDLE_PATH.is_file():
    old = json.loads(OLD_BUNDLE_PATH.read_text(encoding='utf-8'))
    old_rows = old.get('test_scenarios', old.get('test_results', []))
    old_map = {row['label']: row['metrics']['execution_accuracy'] for row in old_rows}
    comparison = []
    for row in tests:
        base_label = row['label'].removeprefix('optimized-luna-')
        before = old_map.get(base_label)
        after = row['metrics']['execution_accuracy']
        comparison.append({
            'scenario': base_label, 'old_accuracy': before,
            'optimized_accuracy': after,
            'delta_pp': None if before is None else (after - before) * 100,
        })
    comparison_df = pd.DataFrame(comparison)
    display(comparison_df.style.format({
        'old_accuracy': '{:.2%}', 'optimized_accuracy': '{:.2%}', 'delta_pp': '{:+.2f}'
    }, na_rep='—'))
else:
    display(Markdown('Không có bundle baseline cũ để so sánh.'))

,scenario,old_accuracy,optimized_accuracy,delta_pp
0,P1-full-one-shot,40.32%,43.55%,+3.23
1,B4-hybrid-fixed,33.87%,43.55%,+9.68
2,B4-plus-event-seeds,38.71%,48.39%,+9.68
3,B4-plus-event-expansion,32.26%,48.39%,+16.13
4,B5-relational-selector,29.03%,38.71%,+9.68
5,B6-repair-relational,40.32%,45.16%,+4.84
6,B6-repair-hybrid,30.65%,45.16%,+14.52
7,G1-adaptive,33.87%,50.00%,+16.13


## Breakdown theo database và difficulty

In [7]:
RUNS_ROOT = Path(bundle['artifacts']['runtime_root']) / 'runs'
def load_run_rows(run_id: str) -> list[dict]:
    path = RUNS_ROOT / run_id / 'cases.jsonl'
    return [json.loads(line) for line in path.read_text(encoding='utf-8').splitlines() if line.strip()]

raw_records = []
for scenario in tests:
    for row in load_run_rows(scenario['run_id']):
        raw_records.append({
            'scenario_label': scenario['label'],
            'stable_id': row['stable_id'],
            'db_id': row['db_id'],
            'difficulty': row['difficulty'],
            'execution_correct': bool(row.get('execution_correct')),
            'invalid_sql': row.get('prediction_status') not in {'success', 'empty_result'},
            'repair_count': row.get('repair_count', 0),
            'route': row.get('route'),
        })
raw_df = pd.DataFrame(raw_records)
assert len(raw_df) == 8 * 62
by_database = raw_df.groupby(['scenario_label', 'db_id'], as_index=False).agg(
    cases=('stable_id', 'count'), execution_accuracy=('execution_correct', 'mean'),
    invalid_sql_rate=('invalid_sql', 'mean'))
by_difficulty = raw_df.groupby(['scenario_label', 'difficulty'], as_index=False).agg(
    cases=('stable_id', 'count'), execution_accuracy=('execution_correct', 'mean'),
    invalid_sql_rate=('invalid_sql', 'mean'))
display(Markdown('### Theo database'))
display(by_database.style.format({'execution_accuracy': '{:.2%}', 'invalid_sql_rate': '{:.2%}'}))
display(Markdown('### Theo difficulty'))
display(by_difficulty.style.format({'execution_accuracy': '{:.2%}', 'invalid_sql_rate': '{:.2%}'}))

### Theo database

,scenario_label,db_id,cases,execution_accuracy,invalid_sql_rate
0,optimized-luna-B4-hybrid-fixed,debit_card_specializing,30,50.00%,3.33%
1,optimized-luna-B4-hybrid-fixed,financial,32,37.50%,0.00%
2,optimized-luna-B4-plus-event-expansion,debit_card_specializing,30,50.00%,3.33%
3,optimized-luna-B4-plus-event-expansion,financial,32,46.88%,0.00%
4,optimized-luna-B4-plus-event-seeds,debit_card_specializing,30,53.33%,0.00%
5,optimized-luna-B4-plus-event-seeds,financial,32,43.75%,0.00%
6,optimized-luna-B5-relational-selector,debit_card_specializing,30,40.00%,0.00%
7,optimized-luna-B5-relational-selector,financial,32,37.50%,0.00%
8,optimized-luna-B6-repair-hybrid,debit_card_specializing,30,53.33%,0.00%
9,optimized-luna-B6-repair-hybrid,financial,32,37.50%,0.00%


### Theo difficulty

,scenario_label,difficulty,cases,execution_accuracy,invalid_sql_rate
0,optimized-luna-B4-hybrid-fixed,challenging,11,27.27%,9.09%
1,optimized-luna-B4-hybrid-fixed,moderate,34,44.12%,0.00%
2,optimized-luna-B4-hybrid-fixed,simple,17,52.94%,0.00%
3,optimized-luna-B4-plus-event-expansion,challenging,11,45.45%,0.00%
4,optimized-luna-B4-plus-event-expansion,moderate,34,47.06%,2.94%
5,optimized-luna-B4-plus-event-expansion,simple,17,52.94%,0.00%
6,optimized-luna-B4-plus-event-seeds,challenging,11,45.45%,0.00%
7,optimized-luna-B4-plus-event-seeds,moderate,34,47.06%,0.00%
8,optimized-luna-B4-plus-event-seeds,simple,17,52.94%,0.00%
9,optimized-luna-B5-relational-selector,challenging,11,36.36%,0.00%


## Repair diagnostics: first pass, repair success và regression

In [8]:
test_cases = load_benchmark_cases(settings.path(settings.paths.test), 'test')
case_lookup = {case.stable_id: case for case in test_cases}
repair_diagnostics = []
for scenario in [item for item in tests if item['scenario'] == 'B6']:
    rows = load_run_rows(scenario['run_id'])
    detail = []
    for row in rows:
        case = case_lookup[row['stable_id']]
        key = f"test:{case.db_id}"
        initial = row['candidates'][0]['prediction']
        first = evaluate_prediction(case, initial, bundle['artifacts']['database_paths'][key])
        detail.append({
            'first_correct': bool(first['execution_correct']),
            'final_correct': bool(row['execution_correct']),
            'repaired': int(row.get('repair_count', 0)) > 0,
        })
    d = pd.DataFrame(detail)
    attempted = int(d['repaired'].sum())
    repair_diagnostics.append({
        'scenario_label': scenario['label'],
        'cases': len(d),
        'first_pass_accuracy': d['first_correct'].mean(),
        'final_accuracy': d['final_correct'].mean(),
        'repair_attempts': attempted,
        'repair_successes': int((~d['first_correct'] & d['final_correct'] & d['repaired']).sum()),
        'repair_regressions': int((d['first_correct'] & ~d['final_correct'] & d['repaired']).sum()),
    })
repair_df = pd.DataFrame(repair_diagnostics)
display(repair_df.style.format({'first_pass_accuracy': '{:.2%}', 'final_accuracy': '{:.2%}'}))

,scenario_label,cases,first_pass_accuracy,final_accuracy,repair_attempts,repair_successes,repair_regressions
0,optimized-luna-B6-repair-relational,62,45.16%,45.16%,1,0,0
1,optimized-luna-B6-repair-hybrid,62,45.16%,45.16%,0,0,0


## G1 route/trajectory và manifests tái lập

In [9]:
g1 = next(item for item in tests if item['scenario'] == 'G1')
g1_rows = load_run_rows(g1['run_id'])
g1_route_df = pd.DataFrame([{
    'route': row.get('route'),
    'execution_correct': bool(row.get('execution_correct')),
    'trajectory_nodes': ' -> '.join(step.get('node', '?') for step in row.get('trajectory', [])),
} for row in g1_rows])
display(g1_route_df.groupby('route', dropna=False).agg(
    cases=('execution_correct', 'size'),
    execution_accuracy=('execution_correct', 'mean'),
).reset_index().style.format({'execution_accuracy': '{:.2%}'}))
display(g1_route_df.groupby('trajectory_nodes', as_index=False).size().sort_values('size', ascending=False))

manifest_rows = []
for scenario in tests:
    manifest_path = RUNS_ROOT / scenario['run_id'] / 'manifest.json'
    manifest = json.loads(manifest_path.read_text(encoding='utf-8'))
    assert manifest['model_provider'] == 'openai' and manifest['model'] == 'gpt-5.6-luna'
    manifest_rows.append({
        'label': scenario['label'],
        'run_id': manifest['run_id'],
        'model': manifest['model'],
        'split': manifest['split'],
        'seed': manifest['seed'],
        'dataset_fingerprint': manifest['dataset_fingerprint'],
        'index_fingerprint': manifest['index_fingerprint'],
        'parameters': json.dumps(manifest['parameters'], ensure_ascii=False, sort_keys=True),
    })
manifest_df = pd.DataFrame(manifest_rows)
display(manifest_df)
print('All test manifests: PASS')

,route,cases,execution_accuracy
0,full,14,50.00%
1,hybrid,21,61.90%
2,sag,27,40.74%


,trajectory_nodes,size
1,choose_context_route -> assemble_linked_contex...,27
2,choose_context_route -> assemble_linked_contex...,21
0,choose_context_route -> assemble_full_context ...,14


,label,run_id,model,split,seed,dataset_fingerprint,index_fingerprint,parameters
0,optimized-luna-P1-full-one-shot,test-optimized-luna-P1-full-one-shot-4a9a02240d34,gpt-5.6-luna,test,42,f11b36d52f85128dd46534a768b137dcf516de0745928d...,68f53f487eed75e87e83884327a9277d44d62bc01a37ca...,"{""context"": {}, ""evaluation_protocol"": ""sqlite..."
1,optimized-luna-B4-hybrid-fixed,test-optimized-luna-B4-hybrid-fixed-8e9d7e4b63b0,gpt-5.6-luna,test,42,f11b36d52f85128dd46534a768b137dcf516de0745928d...,68f53f487eed75e87e83884327a9277d44d62bc01a37ca...,"{""context"": {""candidate_k"": 16, ""column_k"": 12..."
2,optimized-luna-B4-plus-event-seeds,test-optimized-luna-B4-plus-event-seeds-6b7b4a...,gpt-5.6-luna,test,42,f11b36d52f85128dd46534a768b137dcf516de0745928d...,68f53f487eed75e87e83884327a9277d44d62bc01a37ca...,"{""context"": {""candidate_k"": 16, ""column_k"": 12..."
3,optimized-luna-B4-plus-event-expansion,test-optimized-luna-B4-plus-event-expansion-7a...,gpt-5.6-luna,test,42,f11b36d52f85128dd46534a768b137dcf516de0745928d...,68f53f487eed75e87e83884327a9277d44d62bc01a37ca...,"{""context"": {""candidate_k"": 16, ""column_k"": 12..."
4,optimized-luna-B5-relational-selector,test-optimized-luna-B5-relational-selector-b38...,gpt-5.6-luna,test,42,f11b36d52f85128dd46534a768b137dcf516de0745928d...,68f53f487eed75e87e83884327a9277d44d62bc01a37ca...,"{""context"": {""candidate_k"": 16, ""column_k"": 12..."
5,optimized-luna-B6-repair-relational,test-optimized-luna-B6-repair-relational-def42...,gpt-5.6-luna,test,42,f11b36d52f85128dd46534a768b137dcf516de0745928d...,68f53f487eed75e87e83884327a9277d44d62bc01a37ca...,"{""context"": {""candidate_k"": 16, ""column_k"": 12..."
6,optimized-luna-B6-repair-hybrid,test-optimized-luna-B6-repair-hybrid-0f6014baa3bf,gpt-5.6-luna,test,42,f11b36d52f85128dd46534a768b137dcf516de0745928d...,68f53f487eed75e87e83884327a9277d44d62bc01a37ca...,"{""context"": {""candidate_k"": 16, ""column_k"": 12..."
7,optimized-luna-G1-adaptive,test-optimized-luna-G1-adaptive-8b4a8db93a9b,gpt-5.6-luna,test,42,f11b36d52f85128dd46534a768b137dcf516de0745928d...,68f53f487eed75e87e83884327a9277d44d62bc01a37ca...,"{""context"": {""candidate_k"": 16, ""column_k"": 12..."


All test manifests: PASS
